# R3-J: Table 5 on the DMP sensor sets of Table 7
Attach the dataset **snapshot-19-11-2026**. CPU, no Internet needed. Run all.

Table 5 was built with the DMP-2 sensor sets (notebook S / N2), whereas Table 7 and the rest of the paper use
the DMP sets recorded by e14 (`K.baselines`, e.g. 634-692-680 on the 13-node feeder). This notebook recomputes
every row of Table 5 on the e14 DMP sets, with the same protocol as Table 7 (FPS, FAST LightGBM, 2 seeds x 5 folds):

* accuracy at every TVE accuracy class of the sweep 0.0025-3 % and at the common reading (1 %, 0.5 deg);
* the largest TVE class that still reaches 90, 95 and 99 %.

Output: `/kaggle/working/R3_results.zip` with `R3_J_table5.json` and `R3_J_table5.csv`.

In [ ]:
# ============================== SETUP ==============================
SMOKE = False            # True = small quick test (minutes)
RUN = dict(A=False, B=False, C=False, D=False, E=False, F=False, G=False, H=False, J=True)  # A-G already done
H_STRATEGIES = ("limit", "kfactor", "kfactor_neg", "mc")  # part H inverter fault strategies
H_PEN = 0.6              # part H: inverter output, share of feeder load
H_IMAX = 1.2             # part H: inverter current limit (pu of rated current)
N_SIM = 12000            # samples per new simulation (parts F, G); same as the original data sets
RF_MAX_NEW = 1000.0      # part F: upper fault resistance (ohm)
DER_LEVELS = (0.3, 0.6)  # part G: inverter DER penetration (share of feeder load)

import os, sys, glob, json, time, shutil, warnings
warnings.filterwarnings("ignore")
t0 = time.time()
_c = sorted(glob.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
assert _c, "Attach the dataset 'snapshot-fd-code-data'"
DATA = os.path.dirname(_c[0])
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results"); os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS; os.environ["K4_DATA"] = DATA
sys.path.insert(0, DATA)

if RUN["F"] or RUN["G"] or RUN.get("H"):
    try:
        import opendssdirect  # noqa
    except ImportError:
        whl = sorted(glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True))
        if whl:   # offline: dataset 'opendss-wheels'
            os.system("pip install -q --no-index --no-deps " + " ".join(f'"{w}"' for w in whl))
        else:     # online: needs Internet ON
            os.system("pip install -q OpenDSSDirect.py")
try:
    import opendssdirect  # noqa
    HAVE_DSS = True
except ImportError:
    import types; sys.modules["opendssdirect"] = types.ModuleType("opendssdirect"); HAVE_DSS = False

import numpy as np, pandas as pd
import k4lib as K, features as F, models as M, placement2 as Q
from sklearn.base import clone
print("DATA", DATA, "| OpenDSS", HAVE_DSS, "| 1% TVE noise", K.NOISE["1% TVE"])

SEEDS = (0,) if SMOKE else (0, 1)
FEEDERS = [13] if SMOKE else [13, 34, 123]
NZ = "1% TVE"

def save(name, obj):
    with open(os.path.join(RESULTS, name + ".json"), "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    print("saved", name)

def small(ds, n=1500):
    # SMOKE: keep a stratified subset
    if not SMOKE: return ds
    idx = np.random.RandomState(0).permutation(ds.n)[:n]
    for a in ("Vpre", "Vf", "Ipre", "If", "ftype", "fbus", "rf", "y"):
        setattr(ds, a, getattr(ds, a)[idx])
    ds.n = len(idx); return ds

def dmp_sets(nb, k=3):
    # the sensor sets used in the manuscript (Table 7), recorded by e14
    b = K.baselines(nb, k)
    if "DMP (proposed)" in b:
        return {"DMP": b["DMP (proposed)"]["buses"], "DMP N-1": b["DMP N-1 (proposed)"]["buses"]}
    raise RuntimeError("e14_placement_baselines.json not found in the dataset")

PAPER_3BUS = ["650", "675", "684"]

In [ ]:
import pandas as pd
TVE_GRID = [0.0025, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 3.0]   # % (P99 TVE class), as in N2
rows = []
for nb in FEEDERS:
    ds = small(K.dataset(nb)); buses = dmp_sets(nb)["DMP"]; obs = K.rows_for_buses(ds, buses)
    levels = [(f"{t}", K.tve_class(t)) for t in TVE_GRID] + [("common reading", K.NOISE["legacy 1%/0.5deg"])]
    for name, noise in levels:
        r = K.folds_acc(K.features_of(ds, obs, 0, noise, "FPS"), ds.y, ds.rf, seeds=SEEDS)
        rows.append(dict(feeder=nb, buses=buses, level=name, sigma_m=noise[0], sigma_a_deg=noise[1],
                         acc=r["acc"], acc_sd=r["acc_sd"], mrf=r["hif"]))
        print(f"IEEE-{nb} {buses} {name:>14s}: {100*r['acc']:.1f} +- {100*r['acc_sd']:.1f}  MRF {100*r['hif']:.1f}", flush=True)
T = pd.DataFrame(rows)
T.to_csv(os.path.join(RESULTS, "R3_J_table5.csv"), index=False)
spec = {}
for nb, g in T[T.level != "common reading"].groupby("feeder"):
    g = g.assign(t=g.level.astype(float))
    spec[int(nb)] = {f"{int(100*q)}%": (float(g[g.acc >= q].t.max()) if (g.acc >= q).any() else None) for q in (0.90, 0.95, 0.99)}
print("\nLargest TVE class (%) that reaches the target accuracy:")
print(pd.DataFrame(spec).to_string())
print("\nAccuracy rows of Table 5 (mean +- sd, %):")
sel = T[T.level.isin(["1.0", "3.0", "common reading"])]
print(sel.pivot_table(index="level", columns="feeder", values=["acc", "acc_sd"]).mul(100).round(1).to_string())
save("R3_J_table5", dict(rows=rows, spec=spec, tve_grid=TVE_GRID, protocol="FPS, FAST LightGBM, seeds (0,1) x 5 folds, e14 DMP sets"))


In [ ]:
shutil.make_archive(os.path.join(WORK, "R3_results"), "zip", RESULTS)
print("wrote /kaggle/working/R3_results.zip   total", round((time.time() - t0) / 3600, 2), "h")